TASK 1 :Loand and inspect

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv("bangalore_tech_salaries.csv")
print("---Initial Inspectcion ---")
print(df.info())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:",df.duplicated().sum())
print("\nShape:",df.shape)

---Initial Inspectcion ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
None

Missing Values:
 Employee_ID         0
Role                0
years_exp           0
Current_CTC         0
Previous_CTC      200
Company             0
company_TYPE        0
Skills             27

TASK 2: Clean

In [3]:

df.columns = df.columns.str.strip().str.lower()
df = df.rename(
    columns={"company_type": "company_type", "education_tier": "education_tier"}
)


df["company_type_clean"] = df["company_type"].str.strip().str.upper()

df["company_type_clean"] = df["company_type_clean"].replace(
    {
        "UNICORN": "Unicorn",
        "MNC/MID-SIZE": "MNC/Mid-size",
        "EARLY-STAGE": "Early-stage",
    }
)


df["education_tier_clean"] = df["education_tier"].astype(str).str.strip()
tier_map = {
    "Tier 1": "Tier 1",
    "T1": "Tier 1",
    "1": "Tier 1",
    "Tier-1": "Tier 1",
    "Tier 2": "Tier 2",
    "T2": "Tier 2",
    "2": "Tier 2",
    "Tier-2": "Tier 2",
    "Tier 3": "Tier 3",
    "T3": "Tier 3",
    "3": "Tier 3",
    "Tier-3": "Tier 3",
}
df["education_tier_clean"] = df["education_tier_clean"].replace(tier_map)


df["role_clean"] = df["role"].astype(str).str.strip()
role_map = {
    "SDE Backend": "SDE Backend",
    "Backend Engineer": "SDE Backend",
    "Backend Dev": "SDE Backend",
    "BE": "SDE Backend",
    "Backend Developer": "SDE Backend",
    "SDE Frontend": "SDE Frontend",
    "Frontend Engineer": "SDE Frontend",
    "FE": "SDE Frontend",
    "SDE Full-Stack": "SDE Full-Stack",
    "Full Stack Dev": "SDE Full-Stack",
    "Data Scientist": "Data Scientist",
    "DS": "Data Scientist",
    "DevOps Engineer": "DevOps Engineer",
    "DevOps": "DevOps",
    "Product Manager": "Product Manager",
    "PM": "Product Manager",
    "Business Analyst": "Business Analyst",
    "BA": "Business Analyst",
    "Data Analyst": "Data Analyst",
    "DA": "Data Analyst",
    "UI/UX Designer": "UI/UX Designer",
}
df["role_clean"] = df["role_clean"].replace(role_map)


df = df.drop_duplicates()
df["skills"] = df["skills"].fillna("").astype(str).str.strip()

In [4]:


def parse_ctc(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().upper()


    if "LPA" in val_str:
        val_str = val_str.replace("LPA", "").strip()


    val_str = val_str.replace(",", "")


    try:
        num = float(val_str)

        if num >= 100000:
            return num / 100000
        return num
    except:
        return np.nan

df["ctc_clean"] = df["current_ctc"].apply(parse_ctc)

df["prev_ctc_clean"] = df["previous_ctc"].apply(parse_ctc)


print(df.dtypes)
print("\nCleaned Roles:\n", df["role_clean"].value_counts())

employee_id              object
role                     object
years_exp                 int64
current_ctc              object
previous_ctc             object
company                  object
company_type             object
skills                   object
location                 object
education_tier           object
joining_year              int64
work_mode                object
company_type_clean       object
education_tier_clean     object
role_clean               object
ctc_clean               float64
prev_ctc_clean          float64
dtype: object

Cleaned Roles:
 role_clean
SDE Backend                  93
Business Analyst             76
Data Scientist               64
Product Manager              55
SDE Frontend                 51
Data Analyst                 50
Site Reliability Engineer    33
Business Systems Analyst     32
Product Lead                 31
ML Engineer                  31
Data Science Engineer        30
Designer                     28
BI Analyst                   2

Q3.1 CTC distribution by role

In [5]:

q31_summary = (
    df.groupby("role_clean")["ctc_clean"]
    .agg(["median", "mean", "min", "max"])
    .sort_values(by="median", ascending=False)
)
print(q31_summary)

                           median       mean   min       max
role_clean                                                  
Product Manager             32.10  36.375510  13.8  80.10000
Sr PM                       30.30  35.218182  10.8  65.50000
ML Engineer                 28.75  31.053571  11.2  64.50000
Product Lead                26.60  28.737036  12.0  43.70000
Frontend Developer          26.15  22.916667  11.0  32.20000
SDE FS                      24.60  29.031818  11.5  70.90000
Fullstack Dev               24.50  20.930769   9.0  27.60000
SRE                         23.90  24.056000   8.8  49.50000
SDE Full-Stack              23.05  26.833333  10.7  71.70000
Data Science Engineer       22.95  26.370833  14.3  75.60000
Business Systems Analyst    22.60  22.918519   6.8  52.20000
UI/UX                       22.00  21.904762   7.0  44.10000
Data Scientist              21.80  26.105660  10.9  73.50000
SDE Frontend                21.50  21.338636   6.7  41.40000
Site Reliability Enginee

Q3.2 Experience curve for SDE backend

In [6]:
sde_backend_df = df[df["role_clean"] == "SDE Backend"].copy()


sde_backend_df["exp_band"] = pd.cut(
    sde_backend_df["years_exp"],
    bins=[-1, 1, 3, 5, 20],
    labels=["0-1 years", "2-3 years", "4-5 years", "6+ years"],
)


q32_summary = (
    sde_backend_df.groupby("exp_band", observed=False)["ctc_clean"]
    .median()
    .to_frame()
)
q32_summary["pct_growth"] = q32_summary["ctc_clean"].pct_change() * 100
print(q32_summary)

           ctc_clean  pct_growth
exp_band                        
0-1 years       11.7         NaN
2-3 years       19.6   67.521368
4-5 years       25.8   31.632653
6+ years        40.2   55.813953


Q3.3: Skill premium for SDEs

In [7]:

sde_mask = df["role_clean"].str.contains(
    "SDE Backend|SDE Frontend|SDE Full-Stack"
)
sde_df = df[sde_mask].copy()

target_skills = ["AWS", "ML", "System Design", "Kubernetes"]
skill_results = {}


for skill in target_skills:
    has_skill = sde_df[
        sde_df["skills"].str.upper().str.contains(skill.upper(), na=False)
    ]
    no_skill = sde_df[
        ~sde_df["skills"].str.upper().str.contains(skill.upper(), na=False)
    ]

    med_with = has_skill["ctc_clean"].median()
    med_without = no_skill["ctc_clean"].median()
    premium_pct = ((med_with - med_without) / med_without) * 100

    skill_results[skill] = {
        "With": med_with,
        "Without": med_without,
        "Premium%": premium_pct,
    }

q33_summary = pd.DataFrame(skill_results).T
print(q33_summary)

               With  Without   Premium%
AWS            23.0    21.30   7.981221
ML             22.0    21.25   3.529412
System Design  22.5    21.00   7.142857
Kubernetes     16.8    21.40 -21.495327


In [8]:
#Q3.4:Company-type premium,same role

In [9]:
comp_premium = (
    df[df["role_clean"] == "SDE Backend"]
    .groupby("company_type_clean")["ctc_clean"]
    .median()
)


uni_median = comp_premium.get("Unicorn", 0)
mnc_median = comp_premium.get("MNC/Mid-size", 0)
unicorn_over_mnc_premium = ((uni_median - mnc_median) / mnc_median) * 100

print(comp_premium)
print(
    f"\nUnicorn premium over MNC for SDE Backend: {unicorn_over_mnc_premium:.2f}%"
)

company_type_clean
Early-stage    15.60
MID-SIZE       18.65
MNC            20.30
Unicorn        26.40
Name: ctc_clean, dtype: float64

Unicorn premium over MNC for SDE Backend: inf%


/tmp/ipykernel_755/1132685616.py:10: RuntimeWarning: divide by zero encountered in scalar divide
  unicorn_over_mnc_premium = ((uni_median - mnc_median) / mnc_median) * 100


 Q3.5 Underpaid professionals

In [10]:

df["exp_band_all"] = pd.cut(
    df["years_exp"],
    bins=[-1, 1, 3, 5, 20],
    labels=["0-1", "2-3", "4-5", "6+"],
)


group_cols = ["role_clean", "company_type_clean", "exp_band_all"]
df["group_size"] = df.groupby(group_cols, observed=False)[
    "ctc_clean"
].transform("count")
reliable_df = df[df["group_size"] >= 10].copy()


reliable_df["group_median"] = reliable_df.groupby(
    group_cols, observed=False
)["ctc_clean"].transform("median")
reliable_df["gap"] = reliable_df["ctc_clean"] - reliable_df["group_median"]


underpaid_top10 = reliable_df.sort_values(by="gap", ascending=True).head(10)
print(
    underpaid_top10[
        [
            "employee_id",
            "role_clean",
            "company_type_clean",
            "years_exp",
            "ctc_clean",
            "group_median",
            "gap",
        ]
    ]
)

    employee_id   role_clean company_type_clean  years_exp  ctc_clean  \
938     BLR0525  SDE Backend                MNC          2       14.8   
94      BLR0328  SDE Backend                MNC          2       16.4   
527     BLR0157  SDE Backend                MNC          2       18.0   
184     BLR0918  SDE Backend                MNC          2       18.0   
876     BLR0965  SDE Backend                MNC          2       18.9   
231     BLR0344  SDE Backend                MNC          3       19.6   
870     BLR0160  SDE Backend                MNC          3       20.3   
241     BLR0725  SDE Backend                MNC          3       20.6   
44      BLR0793  SDE Backend                MNC          3       20.9   
185     BLR0934  SDE Backend                MNC          3       23.3   

     group_median   gap  
938         19.95 -5.15  
94          19.95 -3.55  
527         19.95 -1.95  
184         19.95 -1.95  
876         19.95 -1.05  
231         19.95 -0.35  
870         19

TASK 4 : Build the printed report

In [11]:

b1 = q32_summary.iloc[0]["ctc_clean"]
b2 = q32_summary.iloc[1]["ctc_clean"]
b2_g = q32_summary.iloc[1]["pct_growth"]
b3 = q32_summary.iloc[2]["ctc_clean"]
b3_g = q32_summary.iloc[2]["pct_growth"]
b4 = q32_summary.iloc[3]["ctc_clean"]
b4_g = q32_summary.iloc[3]["pct_growth"]


u_med = comp_premium.get("Unicorn", 0)
m_med = comp_premium.get("MNC/Mid-size", 0)
mid_med = comp_premium.get("Mid-size", 0) if "Mid-size" in comp_premium else 0
early_med = (
    comp_premium.get("Early-stage", 0) if "Early-stage" in comp_premium else 0
)


m_pct = ((m_med - u_med) / u_med) * 100 if u_med else 0
mid_pct = ((mid_med - u_med) / u_med) * 100 if u_med else 0
early_pct = ((early_med - u_med) / u_med) * 100 if u_med else 0



report_header = f"""==
BANGALORE TECH SALARY DECODER
Built by Purajitt | The Unlox Academy | 2-Hour Live Project
===========
========================================
Dataset: 1,000 Bengaluru tech professionals (synthetic)
Period : 2024 employment snapshot

MEDIAN CTC BY ROLE (in LPA)"""

print(report_header)


for role, rows in q31_summary.iterrows():
    print(f"{role:<30} : {rows['median']:>4.1f}")



report_body = f"""
SDE BACKEND CTC BY EXPERIENCE BAND
0 to 1 years      : {b1:>4.1f} LPA median
2 to 3 years      : {b2:>4.1f} LPA median   (+{int(b2_g)}% growth)
4 to 5 years      : {b3:>4.1f} LPA median   (+{int(b3_g)}% growth)
6+ years          : {b4:>4.1f} LPA median   (+{int(b4_g)}% growth)

SKILL PREMIUM FOR SDES (median CTC)
Skill          | With Skill | Without | Premium
AWS            |  {q33_summary.loc['AWS','With']:>4.1f} LPA |  {q33_summary.loc['AWS','Without']:>4.1f}  |  +{int(q33_summary.loc['AWS','Premium%'])}%
ML             |  {q33_summary.loc['ML','With']:>4.1f} LPA |  {q33_summary.loc['ML','Without']:>4.1f}  |  +{int(q33_summary.loc['ML','Premium%'])}%
System Design  |  {q33_summary.loc['System Design','With']:>4.1f} LPA |  {q33_summary.loc['System Design','Without']:>4.1f}  |  +{int(q33_summary.loc['System Design','Premium%'])}%
Kubernetes     |  {q33_summary.loc['Kubernetes','With']:>4.1f} LPA |  {q33_summary.loc['Kubernetes','Without']:>4.1f}  |  +{int(q33_summary.loc['Kubernetes','Premium%'])}%

COMPANY-TYPE PREMIUM (SDE Backend, same role)
Unicorn           : {u_med:>4.1f} LPA <- baseline ceiling
MNC               : {m_med:>4.1f} LPA   ({int(m_pct)}% vs Unicorn)
Mid-size          : {mid_med:>4.1f} LPA   ({int(mid_pct)}% vs Unicorn)
Early-stage       : {early_med:>4.1f} LPA   ({int(early_pct)}% vs Unicorn)

TOP 5 MOST-UNDERPAID PROFESSIONALS"""

print(report_body)



for _, row in underpaid_top10.head(5).iterrows():
    print(
        f"{row['employee_id']:<8}  {row['role_clean']:<15} {row['company_type_clean']:<11} {int(row['years_exp']):>1} yrs    gap: {row['gap']:>4.1f} LPA"
    )

print("=================")

==
BANGALORE TECH SALARY DECODER
Built by Purajitt | The Unlox Academy | 2-Hour Live Project
Dataset: 1,000 Bengaluru tech professionals (synthetic)
Period : 2024 employment snapshot

MEDIAN CTC BY ROLE (in LPA)
Product Manager                : 32.1
Sr PM                          : 30.3
ML Engineer                    : 28.8
Product Lead                   : 26.6
Frontend Developer             : 26.1
SDE FS                         : 24.6
Fullstack Dev                  : 24.5
SRE                            : 23.9
SDE Full-Stack                 : 23.0
Data Science Engineer          : 23.0
Business Systems Analyst       : 22.6
UI/UX                          : 22.0
Data Scientist                 : 21.8
SDE Frontend                   : 21.5
Site Reliability Engineer      : 21.4
SDE Backend                    : 20.9
SDE-Backend                    : 20.6
Infra Engineer                 : 20.5
Full Stack Engineer            : 20.5
DevOps Engineer                : 20.2
UI Designer                 

TASK 5: Three final insight

## KEY INSIGHTS

**1.** **Sr PM** recorded the highest median CTC at **31.50 LPA**, while **Backend Developer** had the lowest median CTC at **16.25 LPA**. Students aiming for higher compensation should build the technical and leadership skills required to transition into senior product management roles.

**2.** Among Software Development Engineers, professionals with **AWS** skills earned approximately **22.25%** higher median CTC than those without AWS. Students should prioritize learning cloud technologies like AWS to improve their salary potential.

**3.** For **SDE Backend** professionals, the median salary increased by approximately **243.59%** from the **0–1 year** experience band to the **6+ years** band. Additionally, **Unicorn** companies paid around **33.74%** higher median CTC than **MNCs** for the same role, making experience growth and targeting high-paying companies valuable career strategies.
